Zadanie 10

In [7]:
from tensorflow import keras
from tensorflow.keras.datasets import imdb

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=20000)

model = keras.Sequential([
    keras.layers.Embedding(input_dim=20000, output_dim=100),
    keras.layers.LSTM(128),
    keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

x_train = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=250)
x_test = keras.preprocessing.sequence.pad_sequences(x_test, maxlen=250)
model.fit(x_train, y_train, epochs=5, batch_size=64, validation_data=(x_test, y_test))

loss, accuracy = model.evaluate(x_test, y_test)
print(f'Test Accuracy: {accuracy:.4f}')

Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 85s 214ms/step - accuracy: 0.7858 - loss: 0.4479 - val_accuracy: 0.8505 - val_loss: 0.3536
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 89s 228ms/step - accuracy: 0.9028 - loss: 0.2479 - val_accuracy: 0.8577 - val_loss: 0.3610
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 89s 227ms/step - accuracy: 0.9247 - loss: 0.1968 - val_accuracy: 0.8619 - val_loss: 0.3311
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 896s 2s/step - accuracy: 0.9568 - loss: 0.1246 - val_accuracy: 0.8411 - val_loss: 0.4104
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 84s 215ms/step - accuracy: 0.9686 - loss: 0.0912 - val_accuracy: 0.8567 - val_loss: 0.4534
782/782 ━━━━━━━━━━━━━━━━━━━━ 27s 34ms/step - accuracy: 0.8567 - loss: 0.4534
Test Accuracy: 0.8567


In [ ]:
import os
import zipfile
import requests
import numpy as np

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


vocab_size = 20000
max_len = 250
embedding_dim = 100

glove_url = "https://nlp.stanford.edu/data/glove.6B.zip"
zip_path = "glove.6B.zip"
glove_path = "glove.6B.100d.txt"


if not os.path.exists(glove_path):
    if not os.path.exists(zip_path):
        print("Pobieram GloVe...")
        response = requests.get(glove_url, stream=True)
        response.raise_for_status()

        with open(zip_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

    print("Rozpakowuję GloVe...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extract(glove_path)

print("Plik GloVe gotowy:", glove_path)


(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


word_index = imdb.get_word_index()

index_to_word = {
    index + 3: word for word, index in word_index.items()
}

index_to_word[0] = "<PAD>"
index_to_word[1] = "<START>"
index_to_word[2] = "<UNK>"
index_to_word[3] = "<UNUSED>"

embeddings_index = {}

print("Wczytuję embeddingi GloVe...")

with open(glove_path, encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = vector

print("Liczba słów w GloVe:", len(embeddings_index))


embedding_matrix = np.zeros((vocab_size, embedding_dim))

hits = 0
misses = 0

for i in range(vocab_size):
    word = index_to_word.get(i)

    if word is not None:
        vector = embeddings_index.get(word)

        if vector is not None:
            embedding_matrix[i] = vector
            hits += 1
        else:
            misses += 1

print("Dopasowane słowa:", hits)
print("Brakujące słowa:", misses)

def build_glove_model(trainable):
    model = Sequential([
        Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim,
            weights=[embedding_matrix],
            input_length=max_len,
            trainable=trainable
        ),

        LSTM(128),

        Dropout(0.5),

        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)


print("\nTrenuję model z GloVe zamrożonym...")

model_glove_frozen = build_glove_model(trainable=False)

history_glove_frozen = model_glove_frozen.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

test_loss_frozen, test_acc_frozen = model_glove_frozen.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("GloVe zamrożony - loss:", test_loss_frozen)
print("GloVe zamrożony - accuracy:", test_acc_frozen)


print("\nTrenuję model z GloVe odblokowanym...")

model_glove_trainable = build_glove_model(trainable=True)

history_glove_trainable = model_glove_trainable.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

test_loss_trainable, test_acc_trainable = model_glove_trainable.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("GloVe odblokowany - loss:", test_loss_trainable)
print("GloVe odblokowany - accuracy:", test_acc_trainable)


print("\n==============================")
print("PORÓWNANIE WYNIKÓW")
print("==============================")
print(f"GloVe zamrożony   trainable=False: accuracy = {test_acc_frozen:.4f}")
print(f"GloVe odblokowany trainable=True:  accuracy = {test_acc_trainable:.4f}")

Pobieram GloVe...
Rozpakowuję GloVe...
Plik GloVe gotowy: glove.6B.100d.txt


c:\Users\joann\Documents\DataScience\LearnIT\CODE\pythonProject\venv2\Lib\site-packages\numpy\lib\_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


X_train: (25000, 250)
X_test: (25000, 250)
Wczytuję embeddingi GloVe...
Liczba słów w GloVe: 400000
Dopasowane słowa: 19128
Brakujące słowa: 872

Trenuję model z GloVe zamrożonym...
Epoch 1/10


c:\Users\joann\Documents\DataScience\LearnIT\CODE\pythonProject\venv2\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


313/313 ━━━━━━━━━━━━━━━━━━━━ 49s 151ms/step - accuracy: 0.6824 - loss: 0.5884 - val_accuracy: 0.7622 - val_loss: 0.5219
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 49s 155ms/step - accuracy: 0.7948 - loss: 0.4529 - val_accuracy: 0.7882 - val_loss: 0.4893
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 50s 161ms/step - accuracy: 0.8196 - loss: 0.3991 - val_accuracy: 0.8346 - val_loss: 0.3873
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 48s 155ms/step - accuracy: 0.8491 - loss: 0.3513 - val_accuracy: 0.8638 - val_loss: 0.3306
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 48s 152ms/step - accuracy: 0.8633 - loss: 0.3230 - val_accuracy: 0.8184 - val_loss: 0.4081
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 49s 156ms/step - accuracy: 0.8723 - loss: 0.3050 - val_accuracy: 0.8700 - val_loss: 0.3095
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 47s 151ms/step - accuracy: 0.8809 - loss: 0.2848 - val_accuracy: 0.8432 - val_loss: 0.3571
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 49s 157ms/step - accuracy: 0.8921 - loss: 0.2662 - val